<a href="https://colab.research.google.com/github/1conto/fiap-cap1-ignitionZero/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ATIVIDADE INTEGRADORA - RELATÓRIO OPERACIONAL DE PRÉ-DECOLAGEM

Quando falamos de telemetria de embarcações, principalmente foguetes, estamos tratando de alguns pontos críticos sobre o sucesso e fracasso da missão. Uma missão bem sucedida precisa iniciar com o pé direito, na decolagem.

Ao longo do desenvolvimento do processo da decolagem da missão Aurora 01, a decolagem precisaria ser extremamente bem sucedida, obtendo sucesso em cada mínimo detalhe, e para essa garantia de sucesso, precisaríamos de uma boa definição dos conceitos necessários para a execução da decolagem com segurança, minimizando os erros.

Além disso, a programação embarcada tem papel crucial para garantir que todos os sensores e módulos estejam funcionando corretamente, e no final garante o sucesso da decolagem.

## Organização da telemetria

O sucesso da missão depende totalmente da organização da telemetria e a definição clara das condições necessárias para a decolagem acontecer.

### Variáveis analisadas

O sucesso da missão depende das seguintes variáveis:
* Temperatura interna;
* Temperatura externa;
* Integridade estrutural (0/1);
* Níveis de energia (%);
* Pressão dos tanques;
* Status dos módulos críticos.

Dentre as variáveis, temos que definir inicialmente qual o tipo que elas assumirão.

No caso das temperaturas internas e externas, temos que assumirá valores numéricos definindo um valor da temperatura, nesse caso usaremos o sistema universal, e consideraremos em Celsius (ºC), dessa forma, as temperaturas irá assumir valores numéricos, como a variação milimétrica de temperatura possui pouca variação e pouca relevância, usaremos valores discretos para as variações de temperatura, então o tipo do dado assumirá valores inteiros (int).

Já para níveis de energia, temos uma carga total na bateria que é calculada para ter uma duração total da missão com um valor de sobra. Dessa forma, precisamos analisar como está a carga dela com base na carga total que ela suporta. Analisando esse conceito, iremos trabalhar com valores percentuais, fazendo a $carga_- atual/carga_-total$, dessa forma, por se tratar de resultado de uma divisão, tratamos o nivel de energia como valores numéricos com casas decimais (float).

Analisando os casos de pressão dos tanques, utilizaremos o sistema métrico como definição da análise da pressão (bar). Nesse método, como os sistemas precisam de métricas milimétricas referente a valores, usaremos para armazenar os dados de pressão os valores numéricos com casas decimais (float).

Os outros pontos como Status dos módulos críticos e Integridade Estrutural, o computador precisa realizar alguns testes para verificar se está íntegro ou se é preciso alguma manutenção em módulos e na integridade. Dessa forma, esses testes só podem receber dois valores do teste, True para caso a estrutura não apresente nenhum problema, e False caso apresente algum problema, assim, temos que esse tipo de variável são booleanos. Os booleanos também podem apresentar valores 0 e 1 para representar False e True respectivamente.


### Módulos Críticos

Quando falamos de verificar a integridade dos módulos críticos, temos que existe algumas características reais entre todos esses módulos. Entretanto, o sucesso da missão demanda conhecermos todos os nossos módulos críticos.

Precisamos reservar em algum lugar da memória um direcionamento dos nossos módulos, para isso, criaremos uma lista com nossos módulos críticos.

Os módulos críticos que consideraremos serão:
* O computador de voo (flight_computer);
* Sistema de navegação (navigation_system_status);
* Sistema de controle de atitude (attitude_control);
* Sistema de comunicação (communication_system);
* Sistema de energia (power_system);
* Sistema de propulsão (propulsion_system).

Cada módulo desse tem um papel importante no processo final de decisão, todos os sistemas precisam estar funcionais para que a missão seja um sucesso, dessa forma, todos os sistemas precisam estar funcionais antes de iniciar a decolagem. Dessa forma, precisamos testar todos os módulos para verificarmos se os módulos estão funcionais antes da decolagem.

Vamos iniciar alocando uma lista com todos os módulos críticos, quando algum módulo crítico precisar ser incluído na lista, basta appendarmos esse módulo na lista.


In [ ]:
modulos_criticos = [ "flight_computer",
    "navigation_system",
    "attitude_control",
    "communication_system",
    "power_system",
    "propulsion_system"]

## Algoritmo de verificação
Criaremos um fluxograma de um algoritmo capaz de classificar se a decolagem está pronta para ser executada ou se precisaremos abortar a decolagem.

### Fluxograma
![Fluxograma](https://github.com/1conto/fiap-cap1-ignitionZero/blob/main/assets/fluxograma-telemetria.svg?raw=1)

Como a análise dos módulos críticos são similares, precisamos receber valores booleanos a respeito dos dados, podemos tratar os dados referentes a esses módulos em uma classe. Com isso, criamos uma classe de módulos críticos, onde colocaremos as características comuns desses módulos.

In [ ]:
import random

class ModulosCriticos:
    def __init__(self, nome):
      self.nome = nome
      self.status = None

    def __str__(self) -> str:
       return {self.nome:self.status}

    def testar_status(self):
        # Define os valores entre True e False com 95% de ser True, e 5% de ser False.
        self.status = random.choices([True, False], weights=[0.95, 0.05])[0]
        return self.status

Usando a mesma lógica do uso de classes para organizarmos os pontos dos Módulos Críticos, usamos também uma classe para organizarmos a estrutura da telemetria. Nessa estrutura de classes, teremos as funções e os atributos que usaremos para a atividade.
Dentre os atributos temos:
* Temperatura Interna e externa: Considerados em °C
* Integridade Estrutural: Avaliado True se estiver integro e False se tiver falhas
* Voltagem bateria: Analisa a tensão da bateria
* Corrente da bateria: Analisa a corrente elétrica em Amperes
* Nivel Bateria: Porcentagem da bateria no momento da leitura
* Capacidade Bateria: Capacidade Total de armazenamento da bateria.
* Energia disponível: Cálculo com os dados para capturarmos a quantidade de energia disponível (kWh)
* Carga Potencia: Valor que vamos precisar durante o período da decolagem
* Perda Energia: Perda esperada de energia no processo da decolagem, avaliado em porcentagem
* Pressão Tanque: Analisa a pressão do momento no tanque
* Modulos: Análise da estrutura dos módulos descritos acima
* Dict Auditoria: Dicionário de motivos explicando cada campo no momento da análise (para auditoria).
* Dict Validações: Dicionário que uso para armazenar os valores das validações de cada campo obrigatório para decolagem.
* Dict Valores: Dicionário para armazenar os valores dos campos que calculamos.
* Decisão Decolagem: String resultante da decisão da telemetria de seguir com o lançamento ou abortar.

Além dos atributos, temos funções de definição de valor para cada campo. Iniciamos cada atributo no valor None e depois associamos os valores, uma vez que não faz sentido definirmos o valor dos atributos já no momento da inicialização da telemetria, visto que um módulo que em um momento inicial estava OK, pode se deteriorar ao longo do processo e não ser mais seguro realizar a decolagem com esse módulo, e devemos abortar a decolagem. Os outros componentes também podem sofrer alterações dos valores.

Além das funções de definição de valores, temos também as funções de validação, onde fazemos as validações se os campos estão em valores seguros para decolagem e autoriza a decolagem ou proíbe. Essas validações altera os valores dos atributos que são dicionários e da decisão de decolagem.

In [ ]:
class Telemetria:
    def __init__(self, list_modulos):
        """
        Nessa camada do __init__ iniciamos todos os atributos que vamos precisar para
        o módulo da Telemetria.
        """
        self.temperatura_interna = None   # Possui
        self.temperatura_externa = None   # Possui
        self.integridade_estrutural = None  # Possui
        self.voltagem_bateria = None
        self.corrente_bateria = None
        self.nivel_bateria = None
        self.capacidade_bateria = None
        self.energia_disponivel = None
        self.carga_potencia = None
        self.perda_energia = None
        self.pressao_tanque = None
        self.modulos = [ModulosCriticos(modulo) for modulo in list_modulos]
        self.dict_validacoes = None
        self.dict_auditoria = None
        self.dict_valores = None
        self.decisao_decolagem = None

    @property
    def valores(self):
        return self.dict_valores

    # Funções para definir valores dos módulos
    def testar_todos_modulos(self):
        for modulo in self.modulos:
          modulo.testar_status()
        return self.modulos

    def captura_temperatura_interna(self):      # Está sendo calculada
        # Distribuição normal com média de 24 e desvio de 6, assim, eu aumento a possibilidade de cair entre o intervalo seguro
        self.temperatura_interna = max(18, min(40, int(random.gauss(24,6))))   # Com essa distribuição, aumento as chances de cair um valor aleatório dentro do limite seguro de lançamento
        return self.temperatura_interna

    def captura_temperatura_externa(self):      # Está sendo calculada
        self.temperatura_externa = max(-10, min(35, int(random.gauss(25,8))))
        return self.temperatura_externa

    def captura_voltagem_bateria(self):       # Adicionada
        # Favor verificar essa distribuição, tentar entender qual distribuição é melhor utilizada
        self.voltagem_bateria = random.uniform(46, 52)
        return self.voltagem_bateria

    def captura_corrente_bateria(self):       # Adicionada
        self.corrente_bateria = random.uniform(20, 120)
        return self.corrente_bateria

    def captura_energy_lvl(self):     # Está sendo calculada
        self.nivel_bateria = random.betavariate(10,1)    # Com essa distribuição aumento as chances de cair entre 80 e 100 de energia.
        return self.nivel_bateria

    def captura_capacidade_bateria(self):     # Adicionada
        self.capacidade_bateria = random.uniform(80, 120)
        return self.capacidade_bateria

    def captura_energia_disponivel(self):     # Adicionada
        self.energia_disponivel = (self.voltagem_bateria * self.capacidade_bateria * self.nivel_bateria) /1000
        return self.energia_disponivel

    def captura_carga_potencia(self):         # Adicionada
        self.carga_potencia = random.uniform(5, 25)
        return self.carga_potencia

    def captura_perda_energia(self):        # Adicionada
        self.perda_energia = random.uniform(2, 8)
        return self.perda_energia

    def captura_pressao_tanque(self):     # Está sendo calculada
        self.pressao_tanque = random.gauss(70,5)
        return self.pressao_tanque

    def captura_integridade_estrutural(self):   # Está sendo calculada
        self.integridade_estrutural = random.choices([True, False], weights=[0.9, 0.1])[0]
        return self.integridade_estrutural

    def captura_infos_telemetria(self):
        self.captura_temperatura_interna()
        self.captura_temperatura_externa()
        self.captura_integridade_estrutural()
        self.captura_energy_lvl()
        self.captura_pressao_tanque()

    def captura_infos_energia(self):
        self.captura_voltagem_bateria()
        self.captura_corrente_bateria()
        self.captura_capacidade_bateria()
        self.captura_carga_potencia()
        self.captura_energia_disponivel()
        self.captura_perda_energia()

    def define_valores_info_energia(self):
        self.captura_infos_energia()
        modulos_energia = ['voltagem_bateria', 'corrente_bateria', 'capacidade_bateria', 'carga_potencia', 'energia_disponivel', 'perda_energia']
        for modulo in modulos_energia:
            valor = getattr(self, modulo)
            self.dict_valores[modulo] = valor


    # Funções de validação
    def validacoes_telemetria(self, valores_seguros):
        self.testar_todos_modulos()
        self.captura_infos_telemetria()
        dict_validacoes = {}
        dict_auditoria = {}
        dict_valores = {}
        for sensor, valor_seguro in valores_seguros.items():
            valor = getattr(self, sensor)
            dict_valores[sensor] = valor
            if sensor == 'modulos':
                dict_validacoes[sensor] = all([modulo.status for modulo in valor])
                dict_valores[sensor] = {modulo.nome: modulo.status for modulo in valor}
                modulos_falhos = [modulo.nome for modulo in valor if not modulo.status]
                dict_auditoria[sensor] = {'valor_atual': all([modulo.status for modulo in valor]), 'regra': f'Os módulos {', '.join(modulos_falhos)}{" não " if len(modulos_falhos)!=0 else ""}estão seguros.'}

            elif isinstance(valor_seguro, dict):
                minimo = valor_seguro['min']
                maximo = valor_seguro['max']
                dict_validacoes[sensor] = minimo <= valor <= maximo
                dict_auditoria[sensor] = {'valor_atual': valor, 'regra': f'{sensor} deve estar entre {minimo} e {maximo}. Valor atual {valor}'}

            elif isinstance(valor_seguro, bool):
                dict_validacoes[sensor] = valor == valor_seguro
                dict_auditoria[sensor] = {'valor_atual': valor, 'regra': f'{sensor} deve ser {valor_seguro}.'}

        self.dict_validacoes = dict_validacoes
        self.dict_auditoria = dict_auditoria
        self.dict_valores = dict_valores
        self.define_valores_info_energia()

    def valida_decolagem(self):
        if all(self.dict_validacoes.values()):
            self.decisao_decolagem = "PRONTO PARA DECOLAR"
        else:
            self.decisao_decolagem = "DECOLAGEM ABORTADA"
        self.dict_valores['Decolagem'] = self.decisao_decolagem
        self.dict_valores['Motivo'] = self.dict_auditoria
        self.dict_valores['Validações'] = self.dict_validacoes
        return self.decisao_decolagem


## Script em Python

Após fazermos a lógica por trás da Telemetria, usamos um dicionário para alimentarmos os valores seguros dos atributos necessários. Dessa forma, garantimos que não existe números mágicos dentro da execução do código e em caso de uma possível mudança nos valores seguros, conseguimos garantir a execução correta do código apenas alterando os valores desse dicionário, é importante esse campo não ser um atributo dentro da camada da classe Telemetria, visto que dependendo do local que vai ocorrer o lançamento, valores seguros para temperatura externa e pressão podem variar. Então, dessa forma, fazemos o código ser reutilizado. Repare que na função validacoes_telemetria onde validamos cada campo, recebemos o dicionário do valor seguro como um parâmetro.

In [ ]:
valores_seguros = {
    "temperatura_interna": {"min": 18, "max": 30},
    "temperatura_externa": {"min": 15, "max": 35},
    "nivel_bateria": {"min": 0.8, "max": 1},
    "pressao_tanque": {"min": 60, "max": 80},
    "integridade_estrutural": True,
    "modulos": True
}

Finalizando todo a a definição dos valores dos campos e dos valores seguros, podemos realizar o teste da nossa Telemetria.

In [ ]:
telemetria_aurora = Telemetria(modulos_criticos)
telemetria_aurora.validacoes_telemetria(valores_seguros)
telemetria_aurora.valida_decolagem()

'DECOLAGEM ABORTADA'

Podemos ver pelo código acima que a definição do status de autorizar ou proibir a decolagem está funcionando. Entretanto, em casos de abortar, não conseguimos identificar se a decolagem foi um falso positivo ou falso negativo. Pensando nisso, dentro da classe de Telemetria existe um encapsulamento dentro da camada valores, que servirá como um processo de auditoria para entendermos se os valores estão condizendo com o resultado que a função valida_decolagem associou.

In [ ]:
telemetria_aurora.valores

{'temperatura_interna': 32,
 'temperatura_externa': 18,
 'nivel_bateria': 0.7880479181569138,
 'pressao_tanque': 74.49948011557828,
 'integridade_estrutural': True,
 'modulos': {'flight_computer': True,
  'navigation_system': True,
  'attitude_control': True,
  'communication_system': True,
  'power_system': True,
  'propulsion_system': True},
 'voltagem_bateria': 47.12280891929977,
 'corrente_bateria': 94.4756215204959,
 'capacidade_bateria': 102.98474843834137,
 'carga_potencia': 10.494456118541235,
 'energia_disponivel': 3.8243418738335966,
 'perda_energia': 2.8161456480608633,
 'Decolagem': 'DECOLAGEM ABORTADA',
 'Motivo': {'temperatura_interna': {'valor_atual': 32,
   'regra': 'temperatura_interna deve estar entre 18 e 30. Valor atual 32'},
  'temperatura_externa': {'valor_atual': 18,
   'regra': 'temperatura_externa deve estar entre 15 e 35. Valor atual 18'},
  'nivel_bateria': {'valor_atual': 0.7880479181569138,
   'regra': 'nivel_bateria deve estar entre 0.8 e 1. Valor atual 0.

## Análise energética

## Análise assistida por IA

O primeiro ponto para fazermos uma análise assistida com IA é criar um banco de ocorrências ao longo das execuções, salvaremos esses pontos em um dataframe. E precisamos de várias ocorrências desse modelo para conseguirmos treinar o nosso modelo. Com isso, irei gerar um dataframe com essas ocorrências.

In [ ]:
import pandas as pd

numero_ocorrencias = 1000
telemetria_geracao = Telemetria(modulos_criticos)
list_valores = []
for i in range(numero_ocorrencias):
    telemetria_geracao.validacoes_telemetria(valores_seguros)
    telemetria_geracao.valida_decolagem()
    list_valores.append(telemetria_geracao.valores)

df_ocorrencias = pd.DataFrame(list_valores)
df_ocorrencias.head(10)

,temperatura_interna,temperatura_externa,nivel_bateria,pressao_tanque,integridade_estrutural,modulos,voltagem_bateria,corrente_bateria,capacidade_bateria,carga_potencia,energia_disponivel,perda_energia,Decolagem,Motivo,Validações
0,26,25,0.681537,71.637209,True,"{'flight_computer': True, 'navigation_system':...",47.147229,83.744026,116.769717,12.625421,3.752111,6.787581,DECOLAGEM ABORTADA,"{'temperatura_interna': {'valor_atual': 26, 'r...","{'temperatura_interna': True, 'temperatura_ext..."
1,22,27,0.871434,75.064401,True,"{'flight_computer': True, 'navigation_system':...",48.531495,28.749440,94.294908,20.231265,3.987919,3.858807,DECOLAGEM ABORTADA,"{'temperatura_interna': {'valor_atual': 22, 'r...","{'temperatura_interna': True, 'temperatura_ext..."
2,21,21,0.927514,69.856245,True,"{'flight_computer': True, 'navigation_system':...",47.822665,101.949296,106.211987,16.391535,4.711159,5.380113,PRONTO PARA DECOLAR,"{'temperatura_interna': {'valor_atual': 21, 'r...","{'temperatura_interna': True, 'temperatura_ext..."
3,34,12,0.976776,67.547025,True,"{'flight_computer': True, 'navigation_system':...",49.847335,52.582267,100.057082,10.527958,4.871747,6.307753,DECOLAGEM ABORTADA,"{'temperatura_interna': {'valor_atual': 34, 'r...","{'temperatura_interna': False, 'temperatura_ex..."
4,18,30,0.935028,61.505132,True,"{'flight_computer': True, 'navigation_system':...",51.983669,32.770169,119.983175,13.681187,5.831925,7.533210,PRONTO PARA DECOLAR,"{'temperatura_interna': {'valor_atual': 18, 'r...","{'temperatura_interna': True, 'temperatura_ext..."
5,25,34,0.989649,69.961997,True,"{'flight_computer': True, 'navigation_system':...",51.790309,71.565307,116.321687,23.309427,5.961979,4.445275,PRONTO PARA DECOLAR,"{'temperatura_interna': {'valor_atual': 25, 'r...","{'temperatura_interna': True, 'temperatura_ext..."
6,20,22,0.808634,67.231534,True,"{'flight_computer': True, 'navigation_system':...",50.872025,20.536893,80.346292,12.881198,3.305195,4.245705,PRONTO PARA DECOLAR,"{'temperatura_interna': {'valor_atual': 20, 'r...","{'temperatura_interna': True, 'temperatura_ext..."
7,18,26,0.846737,75.332599,True,"{'flight_computer': True, 'navigation_system':...",51.357881,112.443654,119.795464,10.425344,5.209500,2.866277,PRONTO PARA DECOLAR,"{'temperatura_interna': {'valor_atual': 18, 'r...","{'temperatura_interna': True, 'temperatura_ext..."
8,18,18,0.960922,72.044584,True,"{'flight_computer': True, 'navigation_system':...",50.920108,97.171108,113.826263,10.566194,5.569546,2.271563,DECOLAGEM ABORTADA,"{'temperatura_interna': {'valor_atual': 18, 'r...","{'temperatura_interna': True, 'temperatura_ext..."
9,18,35,0.968418,75.276062,True,"{'flight_computer': True, 'navigation_system':...",46.351725,111.342732,91.691042,7.290379,4.115811,7.080932,PRONTO PARA DECOLAR,"{'temperatura_interna': {'valor_atual': 18, 'r...","{'temperatura_interna': True, 'temperatura_ext..."


Podemos perceber que os módulos são de importância para definirmos sobre a decolagem. Então, não faz sentido mantermos a coluna módulos descrita apenas como esse dicionário de nome e valor. Precisamos expandir esses valores.

In [ ]:
df_modulos_expandido = df_ocorrencias['modulos'].apply(pd.Series)
df_ocorrencias = pd.concat([df_ocorrencias.drop('modulos', axis=1), df_modulos_expandido], axis=1)
df_ocorrencias.head()

,temperatura_interna,temperatura_externa,nivel_bateria,pressao_tanque,integridade_estrutural,voltagem_bateria,corrente_bateria,capacidade_bateria,carga_potencia,energia_disponivel,perda_energia,Decolagem,Motivo,Validações,flight_computer,navigation_system,attitude_control,communication_system,power_system,propulsion_system
0,26,25,0.681537,71.637209,True,47.147229,83.744026,116.769717,12.625421,3.752111,6.787581,DECOLAGEM ABORTADA,"{'temperatura_interna': {'valor_atual': 26, 'r...","{'temperatura_interna': True, 'temperatura_ext...",True,True,True,True,True,True
1,22,27,0.871434,75.064401,True,48.531495,28.749440,94.294908,20.231265,3.987919,3.858807,DECOLAGEM ABORTADA,"{'temperatura_interna': {'valor_atual': 22, 'r...","{'temperatura_interna': True, 'temperatura_ext...",True,True,True,False,True,True
2,21,21,0.927514,69.856245,True,47.822665,101.949296,106.211987,16.391535,4.711159,5.380113,PRONTO PARA DECOLAR,"{'temperatura_interna': {'valor_atual': 21, 'r...","{'temperatura_interna': True, 'temperatura_ext...",True,True,True,True,True,True
3,34,12,0.976776,67.547025,True,49.847335,52.582267,100.057082,10.527958,4.871747,6.307753,DECOLAGEM ABORTADA,"{'temperatura_interna': {'valor_atual': 34, 'r...","{'temperatura_interna': False, 'temperatura_ex...",True,True,True,True,True,True
4,18,30,0.935028,61.505132,True,51.983669,32.770169,119.983175,13.681187,5.831925,7.533210,PRONTO PARA DECOLAR,"{'temperatura_interna': {'valor_atual': 18, 'r...","{'temperatura_interna': True, 'temperatura_ext...",True,True,True,True,True,True


Para execução de um bom código de análise de IA precisamos garantir que os dados de fato façam sentido com o que deveria ser usado para treinarmos nosso modelo. Iniciaremos então nossa Análise Exploratória dos Dados (EDA).
### Análise exploratória dos dados


Analisando os pontos referentes ao DataFrame, precisamos realizar uma classificação dos dados entre Pronto para decolar ou abortar decolagem. Vamos inicialmente alterar os valores da coluna Decolagem, assumindo valores de 1 para PRONTO PARA DECOLAR e 0 para DECOLAGEM ABORTADA.

In [ ]:
df_ocorrencias['Decolagem'] = df_ocorrencias['Decolagem'].replace({"DECOLAGEM ABORTADA": 0, "PRONTO PARA DECOLAR": 1})
df_ocorrencias.head()

/tmp/ipykernel_1426/2830728088.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_ocorrencias['Decolagem'] = df_ocorrencias['Decolagem'].replace({"DECOLAGEM ABORTADA": 0, "PRONTO PARA DECOLAR": 1})


,temperatura_interna,temperatura_externa,nivel_bateria,pressao_tanque,integridade_estrutural,voltagem_bateria,corrente_bateria,capacidade_bateria,carga_potencia,energia_disponivel,perda_energia,Decolagem,Motivo,Validações,flight_computer,navigation_system,attitude_control,communication_system,power_system,propulsion_system
0,26,25,0.681537,71.637209,True,47.147229,83.744026,116.769717,12.625421,3.752111,6.787581,0,"{'temperatura_interna': {'valor_atual': 26, 'r...","{'temperatura_interna': True, 'temperatura_ext...",True,True,True,True,True,True
1,22,27,0.871434,75.064401,True,48.531495,28.749440,94.294908,20.231265,3.987919,3.858807,0,"{'temperatura_interna': {'valor_atual': 22, 'r...","{'temperatura_interna': True, 'temperatura_ext...",True,True,True,False,True,True
2,21,21,0.927514,69.856245,True,47.822665,101.949296,106.211987,16.391535,4.711159,5.380113,1,"{'temperatura_interna': {'valor_atual': 21, 'r...","{'temperatura_interna': True, 'temperatura_ext...",True,True,True,True,True,True
3,34,12,0.976776,67.547025,True,49.847335,52.582267,100.057082,10.527958,4.871747,6.307753,0,"{'temperatura_interna': {'valor_atual': 34, 'r...","{'temperatura_interna': False, 'temperatura_ex...",True,True,True,True,True,True
4,18,30,0.935028,61.505132,True,51.983669,32.770169,119.983175,13.681187,5.831925,7.533210,1,"{'temperatura_interna': {'valor_atual': 18, 'r...","{'temperatura_interna': True, 'temperatura_ext...",True,True,True,True,True,True


Agora um outro ponto que temos para conseguirmos executar corretamente o algoritmo de classificação é transformarmos os tipos das colunas em tipos numéricos. Logo, primeiro precisamos entender os tipos de dados que são inseridos dentro de cada coluna. Para isso, podemos usar o dtypes.

In [ ]:
df_ocorrencias.dtypes

,0
temperatura_interna,int64
temperatura_externa,int64
nivel_bateria,float64
pressao_tanque,float64
integridade_estrutural,bool
voltagem_bateria,float64
corrente_bateria,float64
capacidade_bateria,float64
carga_potencia,float64
energia_disponivel,float64


Analisando as colunas, podemos perceber que temos as colunas classificadas como int64, float64, bool e object. O único caso de object não é relevante para o treinamento do modelo que é a coluna de Motivo.  
Essa coluna de Motivo está sendo inserida apenas para conseguirmos analisar o que em cada momento de análise está sendo analisado, servindo de base para uma possível auditoria.  
Fora essa coluna object temos as colunas booleanas, assim, podemos fazer um script que altera os valores dessas colunas para numéricas, alterando 0 para False e 1 para True.

In [ ]:
for columns in df_ocorrencias.columns:
    if df_ocorrencias[columns].dtype == 'bool':
        df_ocorrencias[columns] = df_ocorrencias[columns].astype(int)
df_ocorrencias.head()

,temperatura_interna,temperatura_externa,nivel_bateria,pressao_tanque,integridade_estrutural,voltagem_bateria,corrente_bateria,capacidade_bateria,carga_potencia,energia_disponivel,perda_energia,Decolagem,Motivo,Validações,flight_computer,navigation_system,attitude_control,communication_system,power_system,propulsion_system
0,26,25,0.681537,71.637209,1,47.147229,83.744026,116.769717,12.625421,3.752111,6.787581,0,"{'temperatura_interna': {'valor_atual': 26, 'r...","{'temperatura_interna': True, 'temperatura_ext...",1,1,1,1,1,1
1,22,27,0.871434,75.064401,1,48.531495,28.749440,94.294908,20.231265,3.987919,3.858807,0,"{'temperatura_interna': {'valor_atual': 22, 'r...","{'temperatura_interna': True, 'temperatura_ext...",1,1,1,0,1,1
2,21,21,0.927514,69.856245,1,47.822665,101.949296,106.211987,16.391535,4.711159,5.380113,1,"{'temperatura_interna': {'valor_atual': 21, 'r...","{'temperatura_interna': True, 'temperatura_ext...",1,1,1,1,1,1
3,34,12,0.976776,67.547025,1,49.847335,52.582267,100.057082,10.527958,4.871747,6.307753,0,"{'temperatura_interna': {'valor_atual': 34, 'r...","{'temperatura_interna': False, 'temperatura_ex...",1,1,1,1,1,1
4,18,30,0.935028,61.505132,1,51.983669,32.770169,119.983175,13.681187,5.831925,7.533210,1,"{'temperatura_interna': {'valor_atual': 18, 'r...","{'temperatura_interna': True, 'temperatura_ext...",1,1,1,1,1,1


Após finalizarmos a correção dos dados, podemos iniciar o processo de análise para conseguirmos treinar nosso modelo melhor. Essa análise consegue nos dar uma noção prévia de correlação entre nossos parâmetros.

In [ ]:
df_ocorrencias['Decolagem'].value_counts(normalize = True)

,proportion
Decolagem,
0,0.542
1,0.458


Vamos iniciar retirando colunas que não são usadas no nosso processo para definir o status da decolagem. Esse ponto de retirarmos as colunas que não utilizamos no processo é importante para não termos uma correlação espúria e termos um overfitting nos nossos dados. Após correlacionar variáveis que não possuem valores de causa e efeito. Nesse momento, vou transformar o $df_-ocorrencias$ em um $df_-amostra$ filtrando as colunas que usaremos no processo.

In [ ]:
df_ocorrencias.columns

Index(['temperatura_interna', 'temperatura_externa', 'nivel_bateria',
       'pressao_tanque', 'integridade_estrutural', 'voltagem_bateria',
       'corrente_bateria', 'capacidade_bateria', 'carga_potencia',
       'energia_disponivel', 'perda_energia', 'Decolagem', 'Motivo',
       'Validações', 'flight_computer', 'navigation_system',
       'attitude_control', 'communication_system', 'power_system',
       'propulsion_system'],
      dtype='object')

In [ ]:
df_amostra = df_ocorrencias[['temperatura_interna', 'temperatura_externa', 'nivel_bateria',
                  'pressao_tanque', 'integridade_estrutural', 'flight_computer',
                  'navigation_system', 'attitude_control', 'communication_system',
                  'power_system', 'propulsion_system', 'Decolagem'
              ]]
df_amostra.head()

,temperatura_interna,temperatura_externa,nivel_bateria,pressao_tanque,integridade_estrutural,flight_computer,navigation_system,attitude_control,communication_system,power_system,propulsion_system,Decolagem
0,26,25,0.681537,71.637209,1,1,1,1,1,1,1,0
1,22,27,0.871434,75.064401,1,1,1,1,0,1,1,0
2,21,21,0.927514,69.856245,1,1,1,1,1,1,1,1
3,34,12,0.976776,67.547025,1,1,1,1,1,1,1,0
4,18,30,0.935028,61.505132,1,1,1,1,1,1,1,1


Após o filtro das colunas que usaremos, precisamos analisar como está a divisão dos nossos dados. Não podemos ter uma divisão desigual dos nossos casos de decolagem, para não acontecer overfitting nos nossos sistemas.

In [ ]:
df_amostra['Decolagem'].value_counts(normalize=True)

,proportion
Decolagem,
0,0.542
1,0.458


Conseguimos observar que os casos de abortar uma missão e pronta para decolagem estão com distribuições similares. Vamos aproveitar e analisar se temos alguma instância na telemetria que está dando maiores problemas e está nos retornando mais falhas, para percebermos se nos nossos dados de treinamento possui alguma variável que tem um coeficiente angular que pode ser superestimada no modelo.

In [ ]:
df_abortar = df_ocorrencias[df_ocorrencias['Decolagem'] == 0]

In [ ]:
a = df_abortar['Validações'].apply(pd.Series)

In [ ]:
for column in a.columns:
    print(a[column].value_counts(normalize=True))

temperatura_interna
True     0.754613
False    0.245387
Name: proportion, dtype: float64
temperatura_externa
True     0.821033
False    0.178967
Name: proportion, dtype: float64
nivel_bateria
True     0.843173
False    0.156827
Name: proportion, dtype: float64
pressao_tanque
True     0.915129
False    0.084871
Name: proportion, dtype: float64
integridade_estrutural
True     0.819188
False    0.180812
Name: proportion, dtype: float64
modulos
True     0.54059
False    0.45941
Name: proportion, dtype: float64


Podemos perceber, que na maior parte dos casos, as instâncias estão sendo avaliadas como sucedidas, entretanto, temos os Módulos que estão sendo avaliados com 45.9% de falhas. Mas como essa validação dos módulos engloba vários módulos críticos, vamos separar esses módulos para cada módulo em si nos nossos dados de treinamento, então precisamos fazer também uma análise de distrubuição dentro dos módulos.

In [ ]:
for modulo in modulos_criticos:
    print(df_amostra[modulo].value_counts(normalize=True))

flight_computer
1    0.936
0    0.064
Name: proportion, dtype: float64
navigation_system
1    0.944
0    0.056
Name: proportion, dtype: float64
attitude_control
1    0.959
0    0.041
Name: proportion, dtype: float64
communication_system
1    0.962
0    0.038
Name: proportion, dtype: float64
power_system
1    0.96
0    0.04
Name: proportion, dtype: float64
propulsion_system
1    0.956
0    0.044
Name: proportion, dtype: float64


Na célula acima, conseguimos perceber que todos estão com distribuições semelhantes, o que não dá indícios de uma possível supervalorização de algum parâmetro dentro dos módulos.

A partir daqui, podemos iniciar nossos treinamentos do modelo.

### Treinamento do modelo